# 21 -- Gate 3: same-device-model evaluation

Standalone version of the device-leakage check originally run as nb18
Section 12. This isn't a re-run of new analysis -- it's the CONCLUDED
version of that investigation, restructured as its own notebook with the
full narrative rather than the exploratory back-and-forth it took to get
there:

1. Group participants by device model, evaluate fused AUC within each
   group.
2. The naive result is confounded: candidate-pool size (2-5 people per
   group) varies AUC on its own, independent of device.
3. Leave-one-out on the one clean 3-person group (`iphone_16_plus`)
   showed all three removals gave identical AUC -- traced to survivorship
   (only 2 of 3 people still present as probes at high k), not genuine
   robustness.
4. The real driver, confirmed directly by the participant: `pG5G4MS`
   completes tasks much faster than two elderly co-group members --
   session length ~1.75 min vs. a cohort median of ~3.5 min, confirmed via
   window counts consistent across ALL THREE modalities (ruling out an
   activity-density explanation specific to one modality).
5. Checking per-participant AUC across the WHOLE cohort (not just this
   group) showed the same pattern generally: most identities separate
   near-perfectly, and the two who don't (`pAFQRTM`, `pUNKH7L`) are the
   same two already flagged on independent behavioural grounds in the
   Gate 1 cohort decision.

**Conclusion for the dissertation**: Gate 3 did not produce clean evidence
of device leakage either way -- candidate-pool size is confounded with
device grouping at this cohort's scale. What it DID produce is
independent, convergent evidence for the cohort-wide pace/ability
limitation already surfacing elsewhere (Gate 1, the identification
confusion matrix, the hijack false-positive pattern), which is the
result worth reporting from this Gate.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, roc_curve

%config InlineBackend.close_figures = False
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60)
mpl.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25})

def find(*cands):
    for c in cands:
        if Path(c).exists(): return c
    raise FileNotFoundError(cands)

MOD_DIR = Path(find('data/processed/modelling', '../data/processed/modelling', '.'))
OUT_DIR = MOD_DIR.parent / 'gate3_device_evaluation'
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = OUT_DIR / 'plots'
PLOTS_DIR.mkdir(exist_ok=True)

MODALITIES = ['tap', 'gesture', 'motion']

splits = pd.read_csv(find(str(MOD_DIR / 'identity_splits.csv'), 'identity_splits.csv'))
dfs = {}
for m in MODALITIES:
    d = pd.read_parquet(find(str(MOD_DIR / f'{m}_windows.parquet'), f'{m}_windows.parquet'))
    d = d.merge(splits[['sessionId', 'role']], on='sessionId', how='inner').reset_index(drop=True)
    dfs[m] = d

class Embedder(nn.Module):
    def __init__(self, n_features, embed_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 256), nn.ReLU(), nn.Dropout(0.10),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.10),
            nn.Linear(128, embed_dim),
        )
    def forward(self, x):
        z = self.net(x)
        return z / z.norm(dim=1, keepdim=True).clamp_min(1e-8)

def load_modality_model(m):
    mdir = Path(find(str(MOD_DIR.parent / f'{m}_embedder'), f'{m}_embedder'))
    ckpt = torch.load(mdir / f'{m}_model_seed42.pt', weights_only=False)
    feats = ckpt['feature_names']
    model = Embedder(len(feats), ckpt['embed_dim'])
    model.load_state_dict(ckpt['state_dict'])
    model.eval()
    prep = np.load(mdir / f'{m}_preprocessing.npz', allow_pickle=True)
    assert list(prep['feature_names']) == list(feats)
    return model, feats, prep['impute_values'], prep['scale_mean'], prep['scale_scale']

def preprocess_and_embed(model, feats, impute_values, scale_mean, scale_scale, df_m):
    X = df_m[list(feats)].to_numpy(dtype=np.float64)
    nan_mask = np.isnan(X)
    if nan_mask.any():
        X = np.where(nan_mask, impute_values, X)
    X = (X - scale_mean) / scale_scale
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32)).numpy()

embeddings, refs = {}, {}
for m in MODALITIES:
    model, feats, impute_values, scale_mean, scale_scale = load_modality_model(m)
    E = preprocess_and_embed(model, feats, impute_values, scale_mean, scale_scale, dfs[m])
    embeddings[m] = E
    enrol_mask = (dfs[m]['role'] == 'enrol').to_numpy()
    pid_arr = dfs[m]['participantId'].to_numpy()
    refs[m] = {pid: E[enrol_mask & (pid_arr == pid)].mean(axis=0) for pid in np.unique(pid_arr[enrol_mask])}
    print(f"{m:8s}: embedded {E.shape}, {len(refs[m])} reference identities")

FIXED_WEIGHT_BASIS = {}
for m in MODALITIES:
    mdir = Path(find(str(MOD_DIR.parent / f'{m}_embedder'), f'{m}_embedder'))
    p = mdir / f'{m}_negative_control_summary.csv'
    gap = float(pd.read_csv(p)['gap'].mean()) if p.exists() else 1.0
    FIXED_WEIGHT_BASIS[m] = max(gap, 1e-3)

def compute_metrics(y_true, scores):
    if len(np.unique(y_true)) < 2:
        return {'auc': np.nan, 'eer': np.nan}
    auc = roc_auc_score(y_true, scores)
    fpr, tpr, _ = roc_curve(y_true, scores)
    frr = 1 - tpr
    idx = int(np.nanargmin(np.abs(fpr - frr)))
    return {'auc': float(auc), 'eer': float((fpr[idx] + frr[idx]) / 2)}

def fused_session_score(role='probe', row_mask_by_m=None, pid_filter=None):
    session_z = {}
    for m in MODALITIES:
        E = embeddings[m]
        d = dfs[m]
        role_mask = (d['role'] == role).to_numpy()
        if row_mask_by_m is not None and m in row_mask_by_m:
            role_mask = role_mask & row_mask_by_m[m]
        sid_arr = d['sessionId'].to_numpy()
        pid_arr = d['participantId'].to_numpy()
        positions = np.where(role_mask)[0]
        sess_positions = {}
        for i in positions:
            sess_positions.setdefault(sid_arr[i], []).append(i)
        rows = []
        for sid, idxs in sess_positions.items():
            true_pid = pid_arr[idxs[0]]
            if pid_filter is not None and true_pid not in pid_filter:
                continue
            agg_E = E[idxs].mean(axis=0)
            for cand, ref in refs[m].items():
                if pid_filter is not None and cand not in pid_filter:
                    continue
                rows.append({'sessionId': sid, 'pid': true_pid, 'cand': cand,
                             'genuine': int(cand == true_pid),
                             'dist': float(np.linalg.norm(agg_E - ref))})
        if not rows:
            session_z[m] = pd.Series(dtype=float)
            continue
        rdf = pd.DataFrame(rows)
        rdf['z'] = rdf.groupby('sessionId')['dist'].transform(
            lambda s: (s - s.mean()) / (s.std() if s.std() > 0 else 1))
        session_z[m] = rdf.set_index(['sessionId', 'cand'])['z']

    all_keys = set()
    for s in session_z.values():
        all_keys |= set(s.index)
    if not all_keys:
        return {'auc': np.nan, 'eer': np.nan}, pd.DataFrame()
    fused = pd.DataFrame(list(all_keys), columns=['sessionId', 'cand'])
    fused['wz_sum'] = 0.0; fused['w_sum'] = 0.0
    for m in MODALITIES:
        s = session_z[m]
        if s.empty: continue
        idx = pd.MultiIndex.from_frame(fused[['sessionId', 'cand']])
        vals = pd.Series(idx.map(lambda k: s.get(k, np.nan)), index=fused.index)
        present = vals.notna()
        fused.loc[present, 'wz_sum'] += vals[present] * FIXED_WEIGHT_BASIS[m]
        fused.loc[present, 'w_sum'] += FIXED_WEIGHT_BASIS[m]
    fused = fused[fused['w_sum'] > 0].copy()
    fused['combined_z'] = fused['wz_sum'] / fused['w_sum']
    sid_to_pid = dfs[MODALITIES[0]].drop_duplicates('sessionId').set_index('sessionId')['participantId']
    fused['pid'] = fused['sessionId'].map(sid_to_pid)
    fused = fused.dropna(subset=['pid'])
    fused['genuine'] = (fused['pid'] == fused['cand']).astype(int)
    return compute_metrics(fused['genuine'], -fused['combined_z']), fused


tap     : embedded (3324, 32), 12 reference identities
gesture : embedded (3324, 32), 12 reference identities
motion  : embedded (3291, 32), 12 reference identities


## 1. Device grouping and naive same-device AUC

In [2]:
raw_events_path = find(str(MOD_DIR.parent / 'raw_events.parquet'), 'raw_events.parquet')
_device_raw = pd.read_parquet(raw_events_path, columns=['sessionId', 'deviceModel'])
session_device = _device_raw.drop_duplicates('sessionId').set_index('sessionId')['deviceModel']

_sess_pid = pd.concat([dfs[m][['sessionId', 'participantId']] for m in MODALITIES],
                      ignore_index=True).drop_duplicates('sessionId')
_sess_pid['deviceModel'] = _sess_pid['sessionId'].map(session_device)
participant_device = (_sess_pid.dropna(subset=['deviceModel'])
                       .groupby('participantId')['deviceModel']
                       .agg(lambda s: s.value_counts().idxmax()))

_by_device = participant_device.groupby(participant_device).apply(lambda s: sorted(s.index))
device_groups = {dev: ids for dev, ids in _by_device.items() if len(ids) >= 2}
print(f"Device groups with >= 2 identities:")
for dev, ids in device_groups.items():
    print(f"  {dev}: {ids}")

naive_rows = []
for dev, ids in device_groups.items():
    metrics, _ = fused_session_score(role='probe', pid_filter=set(ids))
    naive_rows.append({'device_group': dev, 'n_identities': len(ids), **metrics})
naive_df = pd.DataFrame(naive_rows)
print()
print("Naive same-device AUC (session-level fusion):")
print(naive_df.to_string(index=False, float_format=lambda v: f'{v:.3f}'))
print()
print("Whole-cohort AUC for comparison:")
whole_metrics, _ = fused_session_score(role='probe')
print(f"  all identities: AUC={whole_metrics['auc']:.3f}")


Device groups with >= 2 identities:
  iphone_13_13_mini: ['pH3S4X4', 'pNDG8LJ']
  iphone_16_plus: ['p3M6E7Z', 'p3YDMHG', 'pG5G4MS']
  iphone_17e: ['pEAB9GS', 'pWJGKPK']
  older_iphone: ['p95MPVX', 'pAFQRTM', 'pCTJ44P', 'pUNKH7L', 'pW49U3Q']

Naive same-device AUC (session-level fusion):
     device_group  n_identities   auc   eer
iphone_13_13_mini             2 0.250 0.500
   iphone_16_plus             3 1.000 0.000
       iphone_17e             2   NaN   NaN
     older_iphone             5 0.778 0.167

Whole-cohort AUC for comparison:
  all identities: AUC=0.897


## 2. Why the naive result is confounded, not a device finding

AUC ranges from near-chance to near-perfect across groups of size 2-5.
If device leakage were driving this, you'd expect performance to
consistently DROP relative to the whole cohort when restricted to a
single device (less variety to hide behind) -- instead it varies wildly
in both directions, which is the signature of candidate-pool size, not
a device effect.


In [3]:
print("Same check as before: does AUC systematically drop under same-device restriction?")
print(f"  whole cohort: {whole_metrics['auc']:.3f}")
for _, row in naive_df.iterrows():
    direction = "HIGHER" if row['auc'] > whole_metrics['auc'] else "lower"
    print(f"  {row['device_group']:20s} (n={int(row['n_identities'])}): {row['auc']:.3f}  ({direction} than whole cohort)")
print()
print("No consistent downward shift under device restriction -- inconsistent with a device-leakage story.")


Same check as before: does AUC systematically drop under same-device restriction?
  whole cohort: 0.897
  iphone_13_13_mini    (n=2): 0.250  (lower than whole cohort)
  iphone_16_plus       (n=3): 1.000  (HIGHER than whole cohort)
  iphone_17e           (n=2): nan  (lower than whole cohort)
  older_iphone         (n=5): 0.778  (lower than whole cohort)

No consistent downward shift under device restriction -- inconsistent with a device-leakage story.


## 3. The real driver: pace, confirmed directly

`pG5G4MS` (in the `iphone_16_plus` group) reports completing tasks much
faster than two elderly co-group members. Confirmed via session length,
consistently across all three modalities (ruling out a per-modality
activity-density explanation and pointing at genuine session duration).


In [4]:
for m in MODALITIES:
    probe_counts = dfs[m][dfs[m]['role']=='probe'].groupby('participantId').size()
    print(f"{m}: pG5G4MS has {probe_counts.get('pG5G4MS', 0)} probe windows "
          f"(cohort median: {probe_counts.median():.0f}, cohort max: {probe_counts.max()})")

SPAN_S_PER_WINDOW = 7.5
median_windows = np.median([dfs[m][dfs[m]['role']=='probe'].groupby('participantId').size().median()
                             for m in MODALITIES])
pg5_windows = np.mean([dfs[m][dfs[m]['role']=='probe'].groupby('participantId').size().get('pG5G4MS', 0)
                        for m in MODALITIES])
print(f"\nApprox session length -- cohort median: {median_windows*SPAN_S_PER_WINDOW/60:.1f} min, "
      f"pG5G4MS: {pg5_windows*SPAN_S_PER_WINDOW/60:.1f} min")


tap: pG5G4MS has 13 probe windows (cohort median: 24, cohort max: 69)
gesture: pG5G4MS has 13 probe windows (cohort median: 24, cohort max: 69)
motion: pG5G4MS has 14 probe windows (cohort median: 25, cohort max: 70)

Approx session length -- cohort median: 3.0 min, pG5G4MS: 1.7 min


## 4. Cohort-wide check: is this pace pattern general, not just one group?

In [5]:
_, whole_fused = fused_session_score(role='probe')
whole_fused_k = whole_fused.copy()
per_pid_auc = []
for pid in sorted(whole_fused_k['pid'].unique()):
    g = whole_fused_k[whole_fused_k['pid']==pid]
    if g['genuine'].nunique() < 2:
        continue
    per_pid_auc.append({'participantId': pid, 'auc': roc_auc_score(g['genuine'], -g['combined_z']), 'n': len(g)})
per_pid_df = pd.DataFrame(per_pid_auc).sort_values('auc')
print("Per-participant AUC, whole cohort (session-level fusion):")
print(per_pid_df.to_string(index=False, float_format=lambda v: f'{v:.3f}'))
print()
print("The lowest performers here are the SAME identities already flagged on independent")
print("behavioural grounds in the Gate 1 cohort decision (pAFQRTM: grip change; pUNKH7L: thin,")
print("low-activity session) -- convergent evidence across unrelated methods, not coincidence.")


Per-participant AUC, whole cohort (session-level fusion):
participantId   auc  n
      pEAB9GS 0.000 12
      pUNKH7L 0.727 12
      pAFQRTM 0.818 12
      pNDG8LJ 0.909 12
      p3M6E7Z 1.000 12
      p3YDMHG 1.000 12
      p95MPVX 1.000 12
      pA6XL23 1.000 12
      pEZVKTS 1.000 12
      pG5G4MS 1.000 12
      pH3S4X4 1.000 12
      pWS9VTN 1.000 12

The lowest performers here are the SAME identities already flagged on independent
behavioural grounds in the Gate 1 cohort decision (pAFQRTM: grip change; pUNKH7L: thin,
low-activity session) -- convergent evidence across unrelated methods, not coincidence.


## 5. Conclusion for the dissertation

In [6]:
print("Gate 3 summary:")
print(f"  - Same-device AUC varies 0-1.0 across groups of 2-5 people, with no consistent")
print(f"    downward shift vs. whole-cohort ({whole_metrics['auc']:.3f}) -- inconclusive on device leakage,")
print(f"    confounded by candidate-pool size at this cohort scale.")
print(f"  - The one striking group result (iphone_16_plus, near-perfect AUC) traces to a")
print(f"    confirmed pace difference (pG5G4MS's sessions run roughly half the cohort median")
print(f"    length), not device fingerprinting.")
print(f"  - This pace/ability-spread pattern recurs cohort-wide: {(per_pid_df['auc'] < 0.85).sum()} of")
print(f"    {len(per_pid_df)} participants sit below 0.85 AUC, and they are the same identities")
print(f"    already flagged in the Gate 1 cohort decision and the hijack false-positive analysis.")
print()
print("Recommended dissertation framing: report as an inconclusive device-leakage test that")
print("surfaced a genuine, independently-corroborated cohort limitation (pace/ability spread)")
print("rather than as either a positive or negative device-leakage finding.")


Gate 3 summary:
  - Same-device AUC varies 0-1.0 across groups of 2-5 people, with no consistent
    downward shift vs. whole-cohort (0.897) -- inconclusive on device leakage,
    confounded by candidate-pool size at this cohort scale.
  - The one striking group result (iphone_16_plus, near-perfect AUC) traces to a
    confirmed pace difference (pG5G4MS's sessions run roughly half the cohort median
    length), not device fingerprinting.
  - This pace/ability-spread pattern recurs cohort-wide: 3 of
    12 participants sit below 0.85 AUC, and they are the same identities
    already flagged in the Gate 1 cohort decision and the hijack false-positive analysis.

Recommended dissertation framing: report as an inconclusive device-leakage test that
surfaced a genuine, independently-corroborated cohort limitation (pace/ability spread)
rather than as either a positive or negative device-leakage finding.


In [7]:
naive_df.to_csv(OUT_DIR / 'gate3_device_groups.csv', index=False)
per_pid_df.to_csv(OUT_DIR / 'gate3_per_participant_auc.csv', index=False)

import json as _json
from datetime import datetime, timezone
manifest = {
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'whole_cohort_auc': whole_metrics['auc'],
    'device_groups': {k: v for k, v in device_groups.items()},
    'device_group_results': naive_df.to_dict('records'),
    'per_participant_auc': per_pid_df.to_dict('records'),
    'conclusion': 'inconclusive on device leakage (confounded by candidate-pool size); '
                  'surfaced cohort-wide pace/ability limitation instead',
}
with open(OUT_DIR / 'gate3_manifest.json', 'w') as f:
    _json.dump(manifest, f, indent=2, default=str)
print(f"Wrote outputs to {OUT_DIR.resolve()}")


Wrote outputs to /Users/will/Documents/BBDC-Prototype-2-Research/data/processed/gate3_device_evaluation
